# Jacobian lens on Qwen3-8B chain of thought — walkthrough

Qwen3-8B is a hybrid reasoning model: in *thinking mode* it emits an explicit chain of thought between `<think>` and `</think>` before the visible answer. This notebook fits/loads a Jacobian lens for it, samples a CoT rollout, reads the lens out **over the reasoning tokens**, and causally intervenes on the trace (steering and coordinate swaps, paper §5).

GPU expectations: bf16 weights are ~16 GB; everything here runs comfortably on a single 40 GB+ GPU.

In [ ]:
import torch
import transformers

import jlens
from jlens.cot import generate_cot
from jlens.examples import EXAMPLES

jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3-8B"
LENS_PATH = "lenses/qwen3-8b_jacobian_lens.pt"  # produced by scripts/fit_lens.py

## 1. Load the model

`jlens.from_hf` wraps the loaded HF model into the `LensModel` interface; the Qwen3 layout (`model.layers` / `model.norm` / `lm_head`) is auto-detected.

In [ ]:
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

## 2. Fit or load the lens

The paper-default fit is 1000 WikiText sequences of 128 tokens (`scripts/fit_lens.py`, shardable across GPUs). Lens quality saturates early — ~100 prompts is already usable — so the cell below fits a quick lens if you don't have one saved. Fitting is calibrated on ordinary web text even though we'll read chat/thinking traces; that matches the paper's protocol (its production lenses are fitted on a pretraining-like corpus and applied to chat transcripts).

In [ ]:
import os

if os.path.exists(LENS_PATH):
    lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
else:
    from jlens.examples import load_wikitext_prompts

    os.makedirs(os.path.dirname(LENS_PATH), exist_ok=True)
    lens = jlens.fit(
        model,
        load_wikitext_prompts(100),
        dim_batch=16,  # the GPU-memory knob: 8 for 40 GB, 16-32 for 80 GB
        max_seq_len=128,
        checkpoint_path=LENS_PATH + ".ckpt",
        checkpoint_every=25,
    )
    lens.save(LENS_PATH)
lens

## 3. Sample a chain-of-thought rollout

`generate_cot` renders the chat template with `enable_thinking=True`, samples with Qwen3's recommended thinking-mode settings (T=0.6, top-p=0.95, top-k=20 — greedy decoding is discouraged for this model), and locates the `<think>` span in the generated tokens. Everything downstream consumes `trace.input_ids` — the exact sampled tokens — never re-encoded text.

In [ ]:
example = next(e for e in EXAMPLES if e.slug == "cot-arithmetic")
print(example.user, "\n")

trace = generate_cot(model, example.user, enable_thinking=True, seed=0, max_new_tokens=1024)
print(f"prompt {trace.prompt_len} toks | think_span={trace.think_span} | answer_span={trace.answer_span}")
print("--- thinking ---\n", trace.thinking_text[:1500])
print("--- answer ---\n", trace.answer_text)

## 4. Slice the reasoning trace

The interactive layer × position page over the completion tokens. Watch for the problem's intermediate quantities (19 marbles/box, 133 in the crate, 124 after removal) surfacing at mid layers *before* they are written — and lingering after. Pinning their tokens adds rank-tracking charts.

In [ ]:
from jlens.vis import build_page, compute_slice, notebook_iframe

pinned = set()
for word in ["19", "133", "124", "7", "9"]:
    for surface in (word, " " + word):
        ids = tokenizer(surface, add_special_tokens=False).input_ids
        if len(ids) == 1:
            pinned.add(ids[0])

slice_data = compute_slice(
    model,
    lens,
    input_ids=trace.input_ids,
    last_n_tokens=trace.total_len - trace.prompt_len,  # completion only
    layer_stride=2,
    mask_display=True,  # word-like tokens read better on Qwen vocabularies
    pinned_token_ids=pinned,
)
page, _, _ = build_page(
    slice_data,
    tokenizer.decode(trace.input_ids[0].tolist()),
    title=example.section,
    description=example.description,
)
notebook_iframe(page)

## 5. J-lens vs logit lens at the answer boundary

Read out the last thinking token — what is the model *about to conclude*? The J-lens should surface the intermediate/bridge content at mid layers where the raw logit lens is still noise (this is the paper's core readout comparison; `scripts/lens_eval.py` runs it systematically over six bundled prompt distributions).

In [ ]:
boundary = max(trace.answer_span[0] - 1, trace.prompt_len)
band = [l for l in jlens.workspace_band(model.n_layers) if l in lens.jacobians]
layers = band[::4]

j_logits, model_logits, _ = lens.apply(model, input_ids=trace.input_ids, layers=layers, positions=[boundary])
logit_lens, _, _ = lens.apply(
    model, input_ids=trace.input_ids, layers=layers, positions=[boundary], use_jacobian=False
)

top5 = lambda logits: [tokenizer.decode([t]) for t in logits.topk(5).indices]
for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(j_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

## 6. Thought injection (additive steering)

Steer the J-direction of an unrelated concept into the workspace band while the model answers an introspective question (paper §7.2). At strength 0 nothing is there; as `alpha` rises the injected concept enters the verbal report. `positions=None` steers every position, including each newly generated token.

In [ ]:
concept = " lightning"
token_id = tokenizer(concept, add_special_tokens=False).input_ids[0]

for alpha in [0.0, 4.0, 8.0]:
    with jlens.steer(model, lens, token_id=token_id, layers=band, alpha=alpha):
        probe = generate_cot(
            model,
            "In one word, what concept is on your mind right now?",
            enable_thinking=False,  # keep the report short
            greedy=True,
            max_new_tokens=12,
        )
    print(f"alpha={alpha:>4}: {probe.answer_text!r}")

## 7. Coordinate swap inside the chain of thought

The projection swap `h ← h + V(σ(c) − c)` exchanges the source/target coordinates while preserving everything orthogonal to their span (§5.4). Swapping the bridge entity of a two-hop question — *Italy → Japan* — across the band at **every** position (the thinking tokens included, since the bridge is formed mid-trace) should flip the reasoning to the counterfactual answer (*euro → yen*).

`scripts/swap_eval.py` runs the systematic 90-item version with an unrelated-pair control.

In [ ]:
question = next(e for e in EXAMPLES if e.slug == "cot-two-hop").user
source_id = tokenizer(" Italy", add_special_tokens=False).input_ids[0]
target_id = tokenizer(" Japan", add_special_tokens=False).input_ids[0]

baseline = generate_cot(model, question, seed=0, max_new_tokens=512)
with jlens.coordinate_swap(
    model, lens, source_token_id=source_id, target_token_id=target_id, layers=band
):
    swapped = generate_cot(model, question, seed=0, max_new_tokens=512)

print("baseline:", baseline.answer_text)
print("swapped: ", swapped.answer_text)
print("\nswapped thinking (first 600 chars):\n", swapped.thinking_text[:600])

### Notes and caveats

- The workspace band (`jlens.workspace_band`: L14–L32 for 36 layers) is the paper's normalized mid-layer range, not something discovered on Qwen3-8B — sweep it for your use case.
- The average Jacobian is a first-order, prompt-averaged approximation; a token in the lens is evidence, not proof, of the concept's causal role — pair readouts with the swap/steer controls above and the unrelated-swap control in `scripts/swap_eval.py`.
- Band-wide, every-position interventions are powerful and off-distribution; track how often unrelated swaps also change the answer before trusting an effect.
- Absence from the lens is not absence from the model: the paper's selectivity results show concepts driving behavior without being verbally broadcast.